In [1]:
import pandas as pd
import s3fs

# Define connection details
minio_endpoint = 'http://localhost:9000'
access_key = 'admin'
secret_key = 'Admin123!'

# Create the filesystem object
fs = s3fs.S3FileSystem(
    key=access_key,
    secret=secret_key,
    client_kwargs={'endpoint_url': minio_endpoint}
)

In [2]:
silver_files = fs.glob("s3://silver/ecommerce_events/**/*.parquet")

silver_df = pd.read_parquet(
    silver_files,
    filesystem=fs,
    engine="pyarrow"
)

In [4]:
silver_df.columns

Index(['source_event_id', 'event_fingerprint', 'event_ts', 'event_year',
       'event_month', 'event_day', 'event_hour', 'event_type', 'product_id',
       'category_id', 'category_code', 'category_l1', 'category_l2',
       'category_l3', 'brand', 'price', 'user_id', 'user_session', 'kafka_ts',
       'kafka_partition', 'kafka_offset', 'bronze_ingested_at',
       'silver_processed_at', 'is_valid', 'invalid_reason', 'event_date'],
      dtype='str')

In [3]:
silver_df['event_fingerprint'].duplicated().any()

np.False_

In [8]:
stg_events_df = silver_df[(silver_df['is_valid'] == True) & (silver_df['event_fingerprint'].notnull())]

# Dedup
stg_events_df = stg_events_df.sort_values(
    by=['bronze_ingested_at', 'kafka_ts', 'kafka_partition', 'kafka_offset'],
    ascending=[False, False, False, False],
    na_position='last'
)
stg_events_df = stg_events_df.drop_duplicates(subset=['event_fingerprint'], keep='first')

In [14]:
stg_events_df.columns

Index(['source_event_id', 'event_fingerprint', 'event_ts', 'event_year',
       'event_month', 'event_day', 'event_hour', 'event_type', 'product_id',
       'category_id', 'category_code', 'category_l1', 'category_l2',
       'category_l3', 'brand', 'price', 'user_id', 'user_session', 'kafka_ts',
       'kafka_partition', 'kafka_offset', 'bronze_ingested_at',
       'silver_processed_at', 'is_valid', 'invalid_reason', 'event_date'],
      dtype='str')